# TRIAGE-EG Notebook 41 — Trial P1 Multimodal Dry Run

Diagnostic-only 24-query run. It creates M0 FULL, M1 FULL and SAFE review candidates, never opens GT, never uploads, and fails closed if any required multimodal input is absent.

In [ ]:
import os
from pathlib import Path
REPO_URL='https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git'
REPO_REF='TRIAGEEG'
REPO_DIR=Path(os.environ.get('AIC_REPO_DIR','/kaggle/working/AIC2026_TeamPTK_SGU'))
RAW_INPUT=Path(os.environ.get('AIC_DATA_ROOT','/kaggle/input/datasets/nadkli/dataset-aic'))
TRIAL_INPUT=Path(os.environ.get('AIC_TRIAL_ROOT','/kaggle/input/datasets/irthn1311/thunghiem-bo-de-thi'))
BCF1_INPUT=Path(os.environ.get('AIC_TRIAL_BCF1_ROOT','/kaggle/input/datasets/irthn1311/trial-p1-true-bcf1-bundle'))
ASR_INPUT=Path(os.environ.get('AIC_ASR_EXTERNAL_V3_ROOT','/kaggle/input/datasets/irthn1311/asr-external-v3-validated'))
EXTERNAL_INPUT=Path(os.environ.get('AIC_EXTERNAL_RUNTIME_EVIDENCE_ROOT','/kaggle/input/datasets/irthn1311/external-multimodal-runtime-evidence-v3'))
E5_INPUT=Path(os.environ.get('AIC_E5_QUERY_ENCODER_ROOT','/kaggle/input/datasets/irthn1311/aic2026-multilingual-e5-small-onnx-query-encoder'))
XCLIP_INPUT=Path(os.environ.get('AIC_XCLIP_ROOT','/kaggle/input/datasets/irthn1311/fs1-xclip-base-patch32-asset'))
QWEN_INPUT=Path(os.environ.get('AIC_QWEN_ROOT','/kaggle/input/datasets/irthn1311/fs1-qwen2-5-vl-3b-instruct-asset'))
OUTPUT_ROOT=Path('/kaggle/working/trial_p1_multimodal_dryrun')
OUTPUT_ZIP=Path('/kaggle/working/trial_p1_multimodal_dryrun_bundle.zip')
WORK_ROOT=Path('/kaggle/working/trial_p1_multimodal_work')
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True); WORK_ROOT.mkdir(parents=True,exist_ok=True)
print({'required_inputs':{'raw_dataset':str(RAW_INPUT),'official_trial_package':str(TRIAL_INPUT),'frozen_true_bcf1':str(BCF1_INPUT),'asr_external_v3_validated':str(ASR_INPUT),'external_ocr_object_runtime_evidence':str(EXTERNAL_INPUT),'e5_query_encoder':str(E5_INPUT),'xclip_offline_asset':str(XCLIP_INPUT),'qwen_offline_asset':str(QWEN_INPUT)},'optional_inputs':{},'internet_required':'GIT_FETCH_AND_PINNED_ONNXRUNTIME_INSTALL_IF_MISSING','model_download_required':False,'gpu':'ONE_T4_FOR_XCLIP_THEN_QWEN_SEQUENTIALLY','gt_opened':False,'output_zip':str(OUTPUT_ZIP)})


In [ ]:
import importlib,subprocess,sys
from importlib import metadata
ONNXRUNTIME_VERSION='1.27.0'
try:
    installed_ort=metadata.version('onnxruntime')
except metadata.PackageNotFoundError:
    installed_ort=None
if installed_ort != ONNXRUNTIME_VERSION:
    subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check','--no-input','--only-binary=:all:',f'onnxruntime=={ONNXRUNTIME_VERSION}'],check=True)
    importlib.invalidate_caches()
import onnxruntime as ort
from tokenizers import Tokenizer
if ort.__version__ != ONNXRUNTIME_VERSION: raise RuntimeError(f'ONNXRUNTIME_VERSION_MISMATCH:{ort.__version__}:{ONNXRUNTIME_VERSION}')
if 'CPUExecutionProvider' not in ort.get_available_providers(): raise RuntimeError(f'ONNXRUNTIME_CPU_PROVIDER_MISSING:{ort.get_available_providers()}')
print({'onnxruntime':ort.__version__,'providers':ort.get_available_providers(),'tokenizers':metadata.version('tokenizers'),'dependency_install_performed':installed_ort != ONNXRUNTIME_VERSION,'model_download_performed':False})


In [ ]:
import subprocess,sys
if not (REPO_DIR/'.git').is_dir(): subprocess.run(['git','clone','--branch',REPO_REF,'--single-branch',REPO_URL,str(REPO_DIR)],check=True)
subprocess.run(['git','fetch','origin',REPO_REF],cwd=REPO_DIR,check=True)
subprocess.run(['git','checkout','--detach','FETCH_HEAD'],cwd=REPO_DIR,check=True)
HEAD=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO_DIR,text=True).strip()
GIT_STATUS=subprocess.check_output(['git','status','--short'],cwd=REPO_DIR,text=True).strip()
sys.path.insert(0,str(REPO_DIR/'src'))
import torch
if not torch.cuda.is_available(): raise RuntimeError('TRIAL_MULTIMODAL_T4_REQUIRED')
print({'source_ref':REPO_REF,'HEAD':HEAD,'checkout_mode':'DETACHED_FETCH_HEAD','git_status':GIT_STATUS or 'CLEAN','gpu':torch.cuda.get_device_name(0)})


In [ ]:
import json,shutil,zipfile,re
def bounded(root,name,max_depth=6):
    root=Path(root); found=[]
    if not root.exists(): return found
    for directory,subdirs,files in os.walk(root):
        current=Path(directory); depth=len(current.relative_to(root).parts)
        subdirs[:]=[] if depth>=max_depth else [item for item in subdirs if item not in {'.cache','blobs','snapshots'}]
        if name in files: found.append(current/name)
    return found
def normalized(value): return re.sub(r'[^a-z0-9]+','',str(value).casefold())
def mount_key(value):
    key=normalized(value)
    for suffix in ('bundle','reports','dataset'):
        if key.endswith(suffix): key=key[:-len(suffix)]
    return key
def roots_for(hint):
    if hint.exists(): return [hint]
    target=mount_key(hint.name); roots=[]
    search=Path('/kaggle/input')
    if search.exists():
        for directory,subdirs,_ in os.walk(search):
            current=Path(directory); depth=len(current.relative_to(search).parts)
            subdirs[:]=[] if depth>=4 else subdirs
            if current.is_dir() and mount_key(current.name)==target: roots.append(current)
    return sorted(set(roots))
def one(root,name):
    root=Path(root); roots=roots_for(root); search_roots=roots or [Path('/kaggle/input')]
    search_depth=6 if roots else 4
    values=sorted(set(path.resolve() for base in search_roots for path in bounded(base,name,search_depth)))
    if not values:
        zip_hits=[]
        for base in search_roots:
            zip_paths=[]
            for directory,subdirs,files in os.walk(base):
                current=Path(directory); depth=len(current.relative_to(base).parts)
                subdirs[:]=[] if depth>=search_depth else subdirs
                zip_paths.extend(current/file for file in files if file.casefold().endswith('.zip'))
            for archive_path in zip_paths:
                with zipfile.ZipFile(archive_path) as archive:
                    members=[member for member in archive.namelist() if Path(member).name==name and not Path(member).is_absolute() and '..' not in Path(member).parts and '\\' not in member]
                    for member in members: zip_hits.append((archive_path,member))
        if len(zip_hits)==1:
            archive_path,member=zip_hits[0]; extract_root=WORK_ROOT/'resolved'/normalized(archive_path.stem); extract_root.mkdir(parents=True,exist_ok=True)
            with zipfile.ZipFile(archive_path) as archive:
                names=archive.namelist()
                if len(names)!=len(set(names)): raise RuntimeError(f'Duplicate ZIP members: {archive_path}')
                for info in archive.infolist():
                    relative=Path(info.filename)
                    if relative.is_absolute() or '..' in relative.parts or '\\' in info.filename: raise RuntimeError(f'Unsafe ZIP member: {info.filename}')
                    target=extract_root/relative
                    if info.is_dir(): target.mkdir(parents=True,exist_ok=True); continue
                    target.parent.mkdir(parents=True,exist_ok=True)
                    with archive.open(info) as source,target.open('wb') as output: shutil.copyfileobj(source,output)
            values=sorted(set(path.resolve() for path in bounded(extract_root,name)))
    if len(values)!=1: raise RuntimeError(f'Expected exactly one {name}; requested={root}; searched={search_roots}; found={values}')
    return values[0]
from triage_eg.trial_p1.multimodal_dryrun import write_blocked_artifacts
required={'trial_mount':TRIAL_INPUT,'query_plans':(BCF1_INPUT,'trial_p1_query_plans_v2.jsonl'),'bcf1_predictions':(BCF1_INPUT,'trial_p1_BCF1_F1_predictions.jsonl'),'asr_provenance':(ASR_INPUT,'asr_external_v3_provenance.json'),'ocr_corpus':(EXTERNAL_INPUT,'ocr_records_external_v3.parquet'),'object_corpus':(EXTERNAL_INPUT,'object_records_external_v3.parquet'),'e5_model':(E5_INPUT,'model.onnx'),'xclip_config':(XCLIP_INPUT,'config.json'),'qwen_config':(QWEN_INPUT,'config.json')}
BLOCKERS=[]; RESOLVED={}
for label,value in required.items():
    try:
        if label=='trial_mount':
            mounts=roots_for(value)
            if len(mounts)!=1: raise RuntimeError(f'Expected one Trial mount; found {mounts}')
            RESOLVED[label]=mounts[0].resolve()
        else: RESOLVED[label]=one(value[0],value[1])
    except Exception as error: BLOCKERS.append(f'{label}: {type(error).__name__}: {error}')
if BLOCKERS:
    write_blocked_artifacts(OUTPUT_ROOT,BLOCKERS,{'HEAD':HEAD,'gt_opened':False,'submission_uploaded':False})
    shutil.make_archive(str(OUTPUT_ZIP.with_suffix('')),'zip',OUTPUT_ROOT)
    raise RuntimeError({'TRIAL_MULTIMODAL_PREFLIGHT_BLOCKED':BLOCKERS,'download_zip':str(OUTPUT_ZIP)})
ASR_ROOT=RESOLVED['asr_provenance'].parent; E5_ROOT=RESOLVED['e5_model'].parent; XCLIP_ROOT=RESOLVED['xclip_config'].parent; QWEN_ROOT=RESOLVED['qwen_config'].parent
print({key:str(value) for key,value in RESOLVED.items()})


In [ ]:
test_env=os.environ.copy(); test_env['PYTHONPATH']=str(REPO_DIR/'src')+(os.pathsep+test_env['PYTHONPATH'] if test_env.get('PYTHONPATH') else '')
test=subprocess.run([sys.executable,'-m','pytest','tests/unit/trial_p1/test_multimodal_dryrun.py','tests/unit/fs1_v11','tests/unit/external_multimodal_v3','-q'],cwd=REPO_DIR,env=test_env,capture_output=True,text=True)
TEST_SUMMARY={'returncode':test.returncode,'stdout_tail':test.stdout.splitlines()[-20:],'stderr_tail':test.stderr.splitlines()[-20:]}
if test.returncode: raise RuntimeError(TEST_SUMMARY)
print(TEST_SUMMARY)


In [ ]:
from triage_eg.trial_p1.multimodal_dryrun import build_asr_candidate_evidence,build_external_parquet_evidence,normalize_trial_plans,read_jsonl
PLANS=read_jsonl(RESOLVED['query_plans']); QUERIES=normalize_trial_plans(PLANS); BCF1=read_jsonl(RESOLVED['bcf1_predictions'])
from triage_eg.trial_p1.asr_v12_loader import ASR_EXTERNAL_V3_SOURCE_TYPE,load_asr_evidence
ASR_LOADER=load_asr_evidence(ASR_ROOT,ASR_EXTERNAL_V3_SOURCE_TYPE)
from triage_eg.external_multimodal_v3.trial_smoke import OnnxE5QueryEncoder,_e5_search
encoder=OnnxE5QueryEncoder(E5_ROOT,exact_revision='03415a4be176a1620747c692ed433219fabc3def')
query_texts=[str(row.get('query','')) for row in QUERIES]
e5_lists=_e5_search(ASR_LOADER,query_texts,encoder,200); E5={row['query_id']:hits for row,hits in zip(QUERIES,e5_lists,strict=True)}
EVIDENCE={'asr':build_asr_candidate_evidence(QUERIES,BCF1,ASR_LOADER,e5_results=E5),'ocr':build_external_parquet_evidence(QUERIES,RESOLVED['ocr_corpus'],'ocr'),'object':build_external_parquet_evidence(QUERIES,RESOLVED['object_corpus'],'object'),'action':{},'qwen':{}}
print({'queries':len(QUERIES),'bcf1_rows':len(BCF1),'asr_rows':sum(map(len,EVIDENCE['asr'].values())),'ocr_rows':sum(map(len,EVIDENCE['ocr'].values())),'object_rows':sum(map(len,EVIDENCE['object'].values())),'e5_encoder_provenance':encoder.provenance})


In [ ]:
from triage_eg.data.stage0_audit.asset_resolver import discover_layout,resolve_assets
from triage_eg.fs1_v11.xclip import XClipAdapter,uniform_indices
from triage_eg.trial_p1.multimodal_dryrun import build_xclip_event_evidence
from triage_eg.video import OpenCVRawVideoDecoder
video_parts,keyframe_parts=discover_layout(RAW_INPUT); xclip=XClipAdapter(XCLIP_ROOT); xclip.load()
def score_window(text,video_id,center):
    assets=resolve_assets(RAW_INPUT,video_id,video_parts,keyframe_parts); decoder=OpenCVRawVideoDecoder(video_id,assets.video)
    start=max(0,min(int(center)-48,decoder.info.total_frames-1)); end=min(decoder.info.total_frames-1,max(start,int(center)+48)); indices=uniform_indices(start,end); frames=[row.image for row in decoder.decode_indices(indices)]; decoder.close(); return xclip.score(text,frames)
EVIDENCE['action']=build_xclip_event_evidence(QUERIES,BCF1,score_window,candidates_per_event=20)
xclip.unload(); print({'xclip_event_rows':sum(map(len,EVIDENCE['action'].values())),'per_trake':{key:len(value) for key,value in EVIDENCE['action'].items()}})


In [ ]:
from PIL import Image
from triage_eg.fs1.qa import GroundingCandidate,bounded_grounding_candidates
from triage_eg.fs1.qwen_adapter import QwenEvidenceAdapter
from triage_eg.fs1_v11.qa import canonical_short_answer,exact_text_variants
from triage_eg.fs1_v11.pipeline import grouped
baseline=grouped(BCF1); qwen=QwenEvidenceAdapter(QWEN_ROOT); qwen.load(); QA_AUDIT=[]
for query in (row for row in QUERIES if row['task']=='QA'):
    query_id=query['query_id']; source=[*EVIDENCE['ocr'][query_id],*EVIDENCE['asr'][query_id],*baseline[query_id]]; candidates=bounded_grounding_candidates([GroundingCandidate(str(row['video_id']),int(row['frame_id']),int(row.get('rank',100)),{'source':row.get('source','bcf1')}) for row in source][:100]); answers=[]
    for candidate in candidates:
        assets=resolve_assets(RAW_INPUT,candidate.video_id,video_parts,keyframe_parts); decoder=OpenCVRawVideoDecoder(candidate.video_id,assets.video); frame=min(candidate.frame_id,decoder.info.total_frames-1); image=Image.fromarray(decoder.decode_indices([frame])[0].image); decoder.close(); context=[row for modality in ('ocr','asr') for row in EVIDENCE[modality][query_id] if row['video_id']==candidate.video_id][:6]; context_text=' | '.join(str(row.get('text') or row.get('asr_span',{}).get('text','')) for row in context); parsed,audit=qwen.answer(candidate,image,description=str(query['query']),question=str(query['query']),evidence_context=context_text,answer_type=str(query.get('answer_type','OTHER')),answer_policy=str(query.get('answer_policy','SHORT_SEMANTIC'))); QA_AUDIT.append({'query_id':query_id,**audit})
        if parsed and parsed.get('evidence_sufficient'):
            kind=str(query.get('answer_type','OTHER')); parsed['answer']=(exact_text_variants(parsed['answer'])[0] if kind=='QUOTE_OR_VISIBLE_TEXT' else canonical_short_answer(parsed['answer'],kind)); answers.append({**parsed,'query_id':query_id,'rank':candidate.evidence_rank,'source':'qwen_bounded_evidence','evidence_context':context_text})
    EVIDENCE['qwen'][query_id]=answers
qwen.unload(); (OUTPUT_ROOT/'qwen_trial_audit.jsonl').write_text(''.join(json.dumps(row,ensure_ascii=False,default=str)+'\n' for row in QA_AUDIT),encoding='utf-8'); print({'qwen_sufficient':{key:len(value) for key,value in EVIDENCE['qwen'].items()}})


In [ ]:
from triage_eg.trial_p1.multimodal_dryrun import build_trial_candidates,sha256_file,write_dryrun_artifacts
def revision_provider(query,event,action):
    rows=[row for row in EVIDENCE['action'].get(query['query_id'],[]) if row.get('event_index')==event.event_index]
    if not rows: raise RuntimeError(f'XCLIP_REVISION_EVIDENCE_MISSING:{query["query_id"]}:{event.event_index}')
    return [{**rows[0],'source':'xclip_graph_revision'}]
RESULT=build_trial_candidates(QUERIES,BCF1,EVIDENCE,revision_provider)
REPORT=write_dryrun_artifacts(OUTPUT_ROOT,QUERIES,BCF1,RESULT,EVIDENCE,causal_fixture_pass=True,provenance={'HEAD':HEAD,'true_bcf1_sha256':sha256_file(RESOLVED['bcf1_predictions']),'asr_source_type':'ASR_EXTERNAL_V3_VALIDATED','e5_query_encoder':encoder.provenance,'gt_opened':False,'submission_uploaded':False,'production_policy_changed':False})
shutil.make_archive(str(OUTPUT_ZIP.with_suffix('')),'zip',OUTPUT_ROOT)
print({'recommendation':REPORT['recommendation'],'hard_trial_gates_pass':REPORT['hard_trial_gates_pass'],'candidate_zips':REPORT['candidate_zips'],'output_zip':str(OUTPUT_ZIP),'output_zip_sha256':sha256_file(OUTPUT_ZIP),'gt_opened':False,'submission_uploaded':False,'cross_l21':'NOT_RUN_UNTIL_HUMAN_REVIEW'})
